In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


PATH_MACRO = Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Original Reproduction\state_z_america.xlsx")

DATASETS = [
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Sectors\Per Region\Sector North America.xlsx"),
     "auto", "Sectors — U.S"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Industries\Developped Countries Industry.xlsx"),
     "auto", "Industries — U.S"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Cross Section Countries\Americas CS Country.xlsx"),
     "auto", "Country Cross Section — America"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Original Reproduction\North_America_6_Factors_Daily_clean.xlsx"),
     "auto", "Factors — U.S."),
]


TRAIN_START, TRAIN_END = pd.Period('2005-01','M'), pd.Period('2018-12','M')
OOS_START,   OOS_END   = pd.Period('2019-01','M'), pd.Period('2024-12','M')


def load_macro_states(path_macro: Path) -> pd.DataFrame:
    Z = pd.read_excel(path_macro)
    if "Month" not in Z.columns:
        for cand in ["month","MONTH","Date","Dates"]:
            if cand in Z.columns:
                Z = Z.rename(columns={cand: "Month"})
                break
    Z["Month"] = pd.to_datetime(Z["Month"], errors="coerce").dt.to_period("M")
    Z = Z.set_index("Month").sort_index()
    macro_cols = [c for c in Z.columns if pd.api.types.is_numeric_dtype(Z[c])]
    return Z[macro_cols].astype(float)

Z_ALL = load_macro_states(PATH_MACRO)


def ann_sr(s: pd.Series) -> float:
    s = pd.Series(s).dropna()
    if s.empty: return np.nan
    sig = s.std()
    return (s.mean()/sig)*np.sqrt(252) if sig != 0 else np.nan

def clean_corr(a: pd.Series, b: pd.Series) -> float:
    df = pd.concat([a, b], axis=1).dropna()
    df = df[(df != 0).any(axis=1)]
    return df.corr().iloc[0,1] if not df.empty else np.nan

def rank_similar(F_full: pd.DataFrame, F_train: pd.DataFrame, cur_per: pd.Period):
    if cur_per not in F_full.index:
        return []
    x0 = F_full.loc[[cur_per]].dropna(axis=1, how="any")
    if x0.empty: return []
    Xtr = F_train.dropna(how="any")
    common = Xtr.columns.intersection(x0.columns)
    if common.empty or Xtr.empty: return []
    X = Xtr[common].values.astype(float)
    x = x0.iloc[0][common].values.astype(float)
    d = np.linalg.norm(X - x[None,:], axis=1)
    return Xtr.index[np.argsort(d)].tolist()

def make_quantiles(ranked_months, n_quantiles: int):
    n = len(ranked_months)
    if n == 0:
        return {q: [] for q in range(1, n_quantiles+1)}
    buckets = np.array_split(ranked_months, n_quantiles)
    return {i+1: list(b) for i,b in enumerate(buckets)}


def load_assets(path: Path, kind: str, min_coverage: float = 1.0):
    df = pd.read_excel(path)

    if "Dates" in df.columns:
        df["Dates"] = pd.to_datetime(df["Dates"], errors="coerce")
        df = df.dropna(subset=["Dates"]).set_index("Dates").sort_index()
        assets = [c for c in df.columns if c != "Dates" and pd.api.types.is_numeric_dtype(df[c])]
        sec_ret_daily   = df[assets].pct_change(fill_method=None).replace([np.inf,-np.inf], np.nan).dropna(how="all")
        sec_ret_monthly = (1.0 + sec_ret_daily).resample("ME").prod() - 1.0

    else:
        std_cols = ["Mkt-RF","SMB","HML","RMW","CMA","Mom","RF"]
        date_col = next((c for c in ["Date","Dates","DATE","date","dt"] if c in df.columns), None)
        present  = [c for c in std_cols if c in df.columns]
        if date_col and len(present) >= 5:
            df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
            df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
            df[present] = df[present].astype(float) / 100.0
            assets = [c for c in present if c != "RF"]
            sec_ret_daily   = df[assets].replace([np.inf,-np.inf], np.nan).dropna(how="all")
            sec_ret_monthly = (1.0 + sec_ret_daily).resample("ME").prod() - 1.0
        else:
            raise ValueError(f"Unsupported schema in {path.name}")

    need = max(1, int(np.ceil(min_coverage * len(assets))))
    mask_good = sec_ret_monthly.notna().sum(axis=1) >= need
    sec_ret_monthly = sec_ret_monthly.loc[mask_good]
    sector_months   = sec_ret_monthly.index.to_period("M")
    keep = sec_ret_daily.index.to_period("M").isin(sector_months)
    sec_ret_daily = sec_ret_daily.loc[keep]
    sec_ret_monthly.index = sec_ret_monthly.index.to_period("M")

    return sec_ret_daily, sec_ret_monthly, assets, sector_months

def build_quantile_series(path: Path, kind: str, n_quantiles: int):
    sec_ret_daily, sec_ret_monthly, assets, sector_months = load_assets(path, kind)

    Z = Z_ALL.reindex(sector_months).ffill()
    F_full  = Z.copy()
    F_train = F_full.loc[(F_full.index >= TRAIN_START) & (F_full.index <= TRAIN_END)].dropna(how="any")

    oos_stop  = min(OOS_END, sector_months.max())
    months_bt = sector_months[(sector_months >= OOS_START) & (sector_months <= oos_stop)]

    returns_by_q = {q: [] for q in range(1, n_quantiles+1)}

    for i in range(len(months_bt)-1):
        cur_per, nxt_per = months_bt[i], months_bt[i+1]
        ranked = rank_similar(F_full, F_train, cur_per)
        if not ranked: continue
        buckets = make_quantiles(ranked, n_quantiles)
        dts = sec_ret_daily.index[sec_ret_daily.index.to_period("M") == nxt_per]
        if len(dts) == 0: continue
        Rm = sec_ret_daily.loc[dts, assets]
        for q in range(1, n_quantiles+1):
            idx_sim = pd.PeriodIndex(buckets[q], freq="M") + 1
            idx_sim = sec_ret_monthly.index.intersection(idx_sim)
            if len(idx_sim) == 0:
                q_daily = pd.Series(0.0, index=dts)
            else:
                signs = {a: np.sign(sec_ret_monthly.loc[idx_sim, a].mean()) for a in assets}
                w = pd.Series(signs, index=assets, dtype=float)
                q_daily = (Rm * w).mean(axis=1)
            returns_by_q[q].extend(q_daily.items())

    q_series = {q: pd.Series(dict(returns_by_q[q])).sort_index() for q in range(1,n_quantiles+1)}
    for q in q_series:
        q_series[q].index = pd.to_datetime(q_series[q].index)
    common_idx = None
    for q in q_series:
        common_idx = q_series[q].index if common_idx is None else common_idx.intersection(q_series[q].index)
    for q in q_series:
        q_series[q] = q_series[q].reindex(common_idx).fillna(0.0)
    long_only = sec_ret_daily[assets].mean(axis=1).reindex(common_idx).fillna(0.0)
    return q_series, long_only

SAVE_DIR = Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Research Notebook\Countries\figures")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

quantile_schemes = [4, 3, 2]  

for n_q in quantile_schemes:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=False)
    axes = axes.ravel()

    for ax, (path, kind, title) in zip(axes, DATASETS):
        q_series, long_only = build_quantile_series(path, kind, n_q)

        for q in range(1, n_q+1):
            sr  = ann_sr(q_series[q])
            ax.plot(q_series[q].index, (1+q_series[q]).cumprod(),
                    label=f"Q{q} (SR {sr:.2f})")

        sr_lo = ann_sr(long_only)
        ax.plot(long_only.index, (1+long_only).cumprod(), color="black", ls="--",
                label=f"Long-only (SR {sr_lo:.2f})")

        ax.set_title(f"{title} — {n_q} quantiles")
        ax.grid(ls=":", alpha=.6)
        ax.legend(fontsize=7, frameon=False)

    plt.suptitle(f"Quantile Portfolios vs Long-only — {n_q} Quantiles", y=0.99)
    plt.tight_layout(rect=[0,0,1,0.96])

    # Save figure
    save_path = SAVE_DIR / f"quantile_{n_q}.png"
    plt.savefig(save_path, dpi=300)
    plt.close(fig)

    print(f"Saved figure: {save_path}")


Saved figure: C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Research Notebook\Countries\figures\quantile_4.png
Saved figure: C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Research Notebook\Countries\figures\quantile_3.png
Saved figure: C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Research Notebook\Countries\figures\quantile_2.png


In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


PATH_MACRO = Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Original Reproduction\state_z_america.xlsx")

DATASETS = [
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Industries\Europe Industries.xlsx"),
     "auto", "Industries — Europe"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Sectors\Per Region\Sector Europe.xlsx"),
     "auto", "Secteur — Europe"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Cross Section Countries\EMEA CS Country.xlsx"),
     "auto", "Country Cross Section — EMEA"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Europe\FF6_daily_factors_cleaned.xlsx"),
     "auto", "Factors — Europe)"),
]
DATASETS = [
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Industries\Emerging Mkt Industries.xlsx"),
     "auto", "Industries — Emerging Markets"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Sectors\Per Region\Sector Asia.xlsx"),
     "auto", "Sectors — Asia"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Cross Section Countries\Asia CS Country.xlsx"),
     "auto", "Country Cross Section — Asia"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Asia\FF6_Asian_daily_cleaned.xlsx"),
     "auto", "Factors — Asia)"),
]

OOS_START, OOS_END = pd.Period('2010-01','M'), pd.Period('2024-12','M')  # ensure 5y lookback


def load_macro_states(path_macro: Path) -> pd.DataFrame:
    Z = pd.read_excel(path_macro)
    if "Month" not in Z.columns:
        for cand in ["month","MONTH","Date","Dates"]:
            if cand in Z.columns:
                Z = Z.rename(columns={cand: "Month"})
                break
    Z["Month"] = pd.to_datetime(Z["Month"], errors="coerce").dt.to_period("M")
    Z = Z.set_index("Month").sort_index()
    macro_cols = [c for c in Z.columns if pd.api.types.is_numeric_dtype(Z[c])]
    return Z[macro_cols].astype(float)

Z_ALL = load_macro_states(PATH_MACRO)


def ann_sr(s: pd.Series) -> float:
    s = pd.Series(s).dropna()
    if s.empty: return np.nan
    sig = s.std()
    return (s.mean()/sig)*np.sqrt(252) if sig != 0 else np.nan

def rank_similar(F_full: pd.DataFrame, F_train: pd.DataFrame, cur_per: pd.Period):
    if cur_per not in F_full.index or F_train.empty:
        return []
    x0 = F_full.loc[[cur_per]].dropna(axis=1, how="any")
    if x0.empty: return []
    Xtr = F_train.dropna(how="any")
    common = Xtr.columns.intersection(x0.columns)
    if common.empty or Xtr.empty: return []
    X = Xtr[common].values.astype(float)
    x = x0.iloc[0][common].values.astype(float)
    d = np.linalg.norm(X - x[None,:], axis=1)
    return Xtr.index[np.argsort(d)].tolist()

def make_quantiles(ranked_months, n_quantiles: int):
    n = len(ranked_months)
    if n == 0:
        return {q: [] for q in range(1, n_quantiles+1)}
    buckets = np.array_split(ranked_months, n_quantiles)
    return {i+1: list(b) for i,b in enumerate(buckets)}


def load_assets(path: Path, kind: str, min_coverage: float = 1.0):
    df = pd.read_excel(path)

    if "Dates" in df.columns:
        df["Dates"] = pd.to_datetime(df["Dates"], errors="coerce")
        df = df.dropna(subset=["Dates"]).set_index("Dates").sort_index()
        assets = [c for c in df.columns if c != "Dates" and pd.api.types.is_numeric_dtype(df[c])]
        sec_ret_daily   = df[assets].pct_change(fill_method=None).replace([np.inf,-np.inf], np.nan).dropna(how="all")
        sec_ret_monthly = (1.0 + sec_ret_daily).resample("ME").prod() - 1.0

    else:
        std_cols = ["Mkt-RF","SMB","HML","RMW","CMA","Mom","RF"]
        date_col = next((c for c in ["Date","Dates","DATE","date","dt"] if c in df.columns), None)
        present  = [c for c in std_cols if c in df.columns]
        if date_col and len(present) >= 5:
            # ---- detect YYYYMMDD format
            if pd.api.types.is_integer_dtype(df[date_col]) or df[date_col].astype(str).str.match(r"^\d{8}$").all():
                df[date_col] = pd.to_datetime(df[date_col].astype(str), format="%Y%m%d", errors="coerce")
            else:
                df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

            df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
            df[present] = df[present].astype(float)
            if df[present].abs().max().max() > 5:
                df[present] /= 100.0

            # Exclude RF
            assets = [c for c in present if c != "RF"]
            sec_ret_daily   = df[assets].replace([np.inf,-np.inf], np.nan).dropna(how="all")
            sec_ret_monthly = (1.0 + sec_ret_daily).resample("ME").prod() - 1.0
        else:
            raise ValueError(f"Unsupported schema in {path.name}")

    need = max(1, int(np.ceil(min_coverage * len(assets))))
    mask_good = sec_ret_monthly.notna().sum(axis=1) >= need
    sec_ret_monthly = sec_ret_monthly.loc[mask_good]
    sector_months   = sec_ret_monthly.index.to_period("M")
    keep = sec_ret_daily.index.to_period("M").isin(sector_months)
    sec_ret_daily = sec_ret_daily.loc[keep]
    sec_ret_monthly.index = sec_ret_monthly.index.to_period("M")

    return sec_ret_daily, sec_ret_monthly, assets, sector_months



def build_quantile_series(path: Path, kind: str, n_quantiles: int):
    sec_ret_daily, sec_ret_monthly, assets, sector_months = load_assets(path, kind)
    Z = Z_ALL.reindex(sector_months).ffill()
    F_full = Z.copy()

    oos_stop  = min(OOS_END, sector_months.max())
    months_bt = sector_months[(sector_months >= OOS_START) & (sector_months <= oos_stop)]

    returns_by_q = {q: [] for q in range(1, n_quantiles+1)}

    for i in range(len(months_bt)-1):
        cur_per, nxt_per = months_bt[i], months_bt[i+1]

        # 5y lookback
        start_train = cur_per - 59
        F_train = F_full.loc[(F_full.index >= start_train) & (F_full.index <= cur_per)].dropna(how="any")

        ranked = rank_similar(F_full, F_train, cur_per)
        if not ranked: continue
        buckets = make_quantiles(ranked, n_quantiles)

        dts = sec_ret_daily.index[sec_ret_daily.index.to_period("M") == nxt_per]
        if len(dts) == 0: continue
        Rm = sec_ret_daily.loc[dts, assets]

        for q in range(1, n_quantiles+1):
            idx_sim = pd.PeriodIndex(buckets[q], freq="M") + 1
            idx_sim = sec_ret_monthly.index.intersection(idx_sim)
            if len(idx_sim) == 0:
                q_daily = pd.Series(0.0, index=dts)
            else:
                signs = {a: np.sign(sec_ret_monthly.loc[idx_sim, a].mean()) for a in assets}
                w = pd.Series(signs, index=assets, dtype=float)
                q_daily = (Rm * w).mean(axis=1)
            returns_by_q[q].extend(q_daily.items())

    q_series = {q: pd.Series(dict(returns_by_q[q])).sort_index() for q in range(1,n_quantiles+1)}
    for q in q_series:
        q_series[q].index = pd.to_datetime(q_series[q].index)

    common_idx = None
    for q in q_series:
        common_idx = q_series[q].index if common_idx is None else common_idx.intersection(q_series[q].index)
    for q in q_series:
        q_series[q] = q_series[q].reindex(common_idx).fillna(0.0)

    long_only = sec_ret_daily[assets].mean(axis=1).reindex(common_idx).fillna(0.0)
    return q_series, long_only

SAVE_DIR = Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Research Notebook\Countries\figures")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

quantile_schemes = [4, 3, 2]

for n_q in quantile_schemes:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=False)
    axes = axes.ravel()

    for ax, (path, kind, title) in zip(axes, DATASETS):
        q_series, long_only = build_quantile_series(path, kind, n_q)

        for q in range(1, n_q+1):
            sr = ann_sr(q_series[q])
            ax.plot(q_series[q].index, (1+q_series[q]).cumprod(),
                    label=f"Q{q} (SR {sr:.2f})")

        sr_lo = ann_sr(long_only)
        ax.plot(long_only.index, (1+long_only).cumprod(), color="black", ls="--",
                label=f"Long-only (SR {sr_lo:.2f})")

        ax.set_title(f"{title} — {n_q} quantiles (5y rolling train)")
        ax.grid(ls=":", alpha=.6)
        ax.legend(fontsize=7, frameon=False)

    plt.suptitle(f"Quantile Portfolios vs Long-only — {n_q} Quantiles\nTraining = Rolling 5 years", y=0.99)
    plt.tight_layout(rect=[0,0,1,0.96])

    save_path = SAVE_DIR / f"quantile_{n_q}_rolling_asia.png"
    plt.savefig(save_path, dpi=300)
    plt.close(fig)

    print(f"Saved figure: {save_path}")


Saved figure: C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Research Notebook\Countries\figures\quantile_4_rolling_asia.png
Saved figure: C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Research Notebook\Countries\figures\quantile_3_rolling_asia.png
Saved figure: C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Research Notebook\Countries\figures\quantile_2_rolling_asia.png


In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


PATH_MACRO = Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Original Reproduction\state_z_america.xlsx")

DATASETS = [
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Sectors\Per Region\Sector North America.xlsx"),
     "auto", "Sectors — U.S"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Industries\Developped Countries Industry.xlsx"),
     "auto", "Industries — U.S"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Cross Section Countries\Americas CS Country.xlsx"),
     "auto", "Country Cross Section — America"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Original Reproduction\North_America_6_Factors_Daily_clean.xlsx"),
     "auto", "Factors — U.S."),
]


OOS_START, OOS_END = pd.Period('2010-01','M'), pd.Period('2024-12','M')

SAVE_DIR = Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Research Notebook\Countries\figures")
SAVE_DIR.mkdir(parents=True, exist_ok=True)


def load_macro_states(path_macro: Path) -> pd.DataFrame:
    Z = pd.read_excel(path_macro)
    if "Month" not in Z.columns:
        for cand in ["month","MONTH","Date","Dates"]:
            if cand in Z.columns:
                Z = Z.rename(columns={cand: "Month"})
                break
    Z["Month"] = pd.to_datetime(Z["Month"], errors="coerce").dt.to_period("M")
    Z = Z.set_index("Month").sort_index()
    macro_cols = [c for c in Z.columns if pd.api.types.is_numeric_dtype(Z[c])]
    return Z[macro_cols].astype(float)

Z_ALL = load_macro_states(PATH_MACRO)

def ann_sr(s: pd.Series) -> float:
    s = pd.Series(s).dropna()
    if s.empty: return np.nan
    sig = s.std()
    return (s.mean()/sig)*np.sqrt(252) if sig != 0 else np.nan

def make_quantiles(ranked_months, n_quantiles: int):
    n = len(ranked_months)
    if n == 0:
        return {q: [] for q in range(1, n_quantiles+1)}
    buckets = np.array_split(ranked_months, n_quantiles)
    return {i+1: list(b) for i,b in enumerate(buckets)}

def load_assets(path: Path, kind: str, min_coverage: float = 1.0):
    df = pd.read_excel(path)

    if "Dates" in df.columns:
        df["Dates"] = pd.to_datetime(df["Dates"], errors="coerce")
        df = df.dropna(subset=["Dates"]).set_index("Dates").sort_index()
        assets = [c for c in df.columns if c != "Dates" and pd.api.types.is_numeric_dtype(df[c])]
        sec_ret_daily   = df[assets].pct_change(fill_method=None).replace([np.inf,-np.inf], np.nan).dropna(how="all")
        sec_ret_monthly = (1.0 + sec_ret_daily).resample("ME").prod() - 1.0
        need = max(1, int(np.ceil(min_coverage * len(assets))))
        mask_good = sec_ret_monthly.notna().sum(axis=1) >= need
        sec_ret_monthly = sec_ret_monthly.loc[mask_good]
        sector_months   = sec_ret_monthly.index.to_period("M")
        keep = sec_ret_daily.index.to_period("M").isin(sector_months)
        sec_ret_daily = sec_ret_daily.loc[keep]
        sec_ret_monthly.index = sec_ret_monthly.index.to_period("M")
        return sec_ret_daily, sec_ret_monthly, assets, sector_months

    ff_cols = ["Mkt-RF","SMB","HML","RMW","CMA","Mom","RF"]
    date_col = next((c for c in ["Date","Dates","DATE","date","dt"] if c in df.columns), None)
    present = [c for c in ff_cols if c in df.columns]
    if date_col and len(present) >= 5:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
        df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()

        df[present] = df[present].astype(float)
        if df[present].abs().max().max() > 5:  
            df[present] /= 100.0

        assets = [c for c in present if c != "RF"]
        sec_ret_daily   = df[assets].replace([np.inf,-np.inf], np.nan).dropna(how="all")
        sec_ret_monthly = (1.0 + sec_ret_daily).resample("ME").prod() - 1.0

        need = max(1, int(np.ceil(min_coverage * len(assets))))
        mask_good = sec_ret_monthly.notna().sum(axis=1) >= need
        sec_ret_monthly = sec_ret_monthly.loc[mask_good]
        sector_months   = sec_ret_monthly.index.to_period("M")

        keep = sec_ret_daily.index.to_period("M").isin(sector_months)
        sec_ret_daily = sec_ret_daily.loc[keep]
        sec_ret_monthly.index = sec_ret_monthly.index.to_period("M")
        return sec_ret_daily, sec_ret_monthly, assets, sector_months

    raise ValueError(f"Unsupported schema in {path.name}")


def euclid_distances(F_train: pd.DataFrame, x0: pd.Series) -> pd.Series:
    common = F_train.columns.intersection(x0.index)
    if len(common) == 0: return pd.Series(dtype=float)
    X = F_train[common].values.astype(float)
    x = x0[common].values.astype(float)
    d = np.linalg.norm(X - x[None,:], axis=1)
    return pd.Series(d, index=F_train.index).sort_values()

def cosine_distances(F_train: pd.DataFrame, x0: pd.Series, eps: float = 1e-12) -> pd.Series:
    common = F_train.columns.intersection(x0.index)
    if len(common) == 0 or F_train.empty:
        return pd.Series(dtype=float)
    X = F_train[common].values.astype(float)
    x = x0[common].values.astype(float).reshape(1,-1)
    x_norm = np.sqrt((x * x).sum(axis=1)) + eps
    X_norm = np.sqrt((X * X).sum(axis=1)) + eps
    cos_sim = (X @ x.T).ravel() / (X_norm * x_norm[0])
    cos_dist = 1.0 - np.clip(cos_sim, -1.0, 1.0)
    return pd.Series(cos_dist, index=F_train.index).sort_values()

def mahalanobis_distances(F_train: pd.DataFrame, x0: pd.Series, ridge: float = 1e-8) -> pd.Series:
    common = F_train.columns.intersection(x0.index)
    if len(common) == 0 or F_train.shape[0] == 0:
        return pd.Series(dtype=float)
    X = F_train[common].values.astype(float)
    x = x0[common].values.astype(float).reshape(1, -1)
    Σ = np.cov(X, rowvar=False)
    Σ = np.atleast_2d(np.asarray(Σ))
    if Σ.shape[0] != Σ.shape[1] or Σ.shape[0] != X.shape[1]:
        var = X.var(axis=0, ddof=1)
        var[var <= 0] = ridge
        Σ = np.diag(var)
    Σ = Σ + ridge * np.eye(Σ.shape[0])
    try:
        Σ_inv = np.linalg.inv(Σ)
    except np.linalg.LinAlgError:
        Σ_inv = np.linalg.pinv(Σ)
    diffs = X - x
    d2 = np.einsum("ij,jk,ik->i", diffs, Σ_inv, diffs)
    d = np.sqrt(np.clip(d2, 0, np.inf))
    return pd.Series(d, index=F_train.index).sort_values()

def build_quantile_series_union(path: Path, kind: str, n_quantiles: int):
    sec_ret_daily, sec_ret_monthly, assets, sector_months = load_assets(path, kind)
    Z = Z_ALL.reindex(sector_months).ffill()
    F_full = Z.copy()

    oos_stop  = min(OOS_END, sector_months.max())
    months_bt = sector_months[(sector_months >= OOS_START) & (sector_months <= oos_stop)]

    returns_by_q = {q: [] for q in range(1, n_quantiles+1)}
    counts_by_q  = {q: [] for q in range(1, n_quantiles+1)}  # track number of similar months

    for i in range(len(months_bt)-1):
        cur_per, nxt_per = months_bt[i], months_bt[i+1]
        start_train = cur_per - 59
        F_train = F_full.loc[(F_full.index >= start_train) & (F_full.index <= cur_per)].dropna(how="any")
        if F_train.empty: 
            continue
        x0 = F_full.loc[cur_per].dropna()
        if x0.empty: 
            continue

        dE = euclid_distances(F_train, x0)
        dC = cosine_distances(F_train, x0)
        dM = mahalanobis_distances(F_train, x0)
        if dE.empty or dC.empty or dM.empty:
            continue

        bE = make_quantiles(list(dE.index), n_quantiles)
        bC = make_quantiles(list(dC.index), n_quantiles)
        bM = make_quantiles(list(dM.index), n_quantiles)

        dts = sec_ret_daily.index[sec_ret_daily.index.to_period("M") == nxt_per]
        if len(dts) == 0: 
            continue
        Rm = sec_ret_daily.loc[dts, assets]

        def mu_next_over(months_set) -> pd.Series:
            if len(months_set) == 0:
                return pd.Series(0.0, index=assets)
            idx_sim_next = (pd.PeriodIndex(list(months_set), freq="M") + 1)
            idx_sim_next = sec_ret_monthly.index.intersection(idx_sim_next)
            if len(idx_sim_next) == 0:
                return pd.Series(0.0, index=assets)
            return sec_ret_monthly.loc[idx_sim_next, assets].mean(axis=0)

        for q in range(1, n_quantiles+1):
            sE, sC, sM = set(bE[q]), set(bC[q]), set(bM[q])
            months_sel = sE | sC | sM   # ✅ union across all quantiles
            mu_final = mu_next_over(months_sel)
            counts_by_q[q].append(len(months_sel))   # track size of union

            signs = np.sign(mu_final.fillna(0.0))
            q_daily = (Rm * signs).mean(axis=1)
            returns_by_q[q].extend(q_daily.items())

    q_series = {q: pd.Series(dict(returns_by_q[q])).sort_index() for q in range(1,n_quantiles+1)}
    for q in q_series:
        q_series[q].index = pd.to_datetime(q_series[q].index)

    common_idx = None
    for q in q_series:
        idx = q_series[q].index
        common_idx = idx if common_idx is None else common_idx.intersection(idx)
    for q in q_series:
        q_series[q] = q_series[q].reindex(common_idx).fillna(0.0)

    long_only = sec_ret_daily[assets].mean(axis=1).reindex(common_idx).fillna(0.0)

    avg_counts = {q: np.mean(counts_by_q[q]) if counts_by_q[q] else 0 for q in counts_by_q}

    return q_series, long_only, avg_counts

quantile_schemes = [4, 3, 2]

for n_q in quantile_schemes:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=False)
    axes = axes.ravel()

    for ax, (path, kind, title) in zip(axes, DATASETS):
        q_series, long_only, avg_counts = build_quantile_series_union(path, kind, n_q)

        for q in range(1, n_q+1):
            sr  = ann_sr(q_series[q])
            ax.plot(q_series[q].index, (1+q_series[q]).cumprod(),
                    label=f"Q{q} (SR {sr:.2f}, mean |months| {avg_counts[q]:.1f})")

        sr_lo = ann_sr(long_only)
        ax.plot(long_only.index, (1+long_only).cumprod(), color="black", ls="--",
                label=f"Long-only (SR {sr_lo:.2f})")

        ax.set_title(f"{title} — {n_q} quantiles (Hybrid dist)")
        ax.grid(ls=":", alpha=.6)
        ax.legend(fontsize=7, frameon=False)

        print(f"{title} — {n_q} quantiles: avg #months per quantile = {avg_counts}")

    plt.suptitle(f"Quantile Portfolios vs Long-only — {n_q} Quantiles\nHybrid Similarity", y=0.99)
    plt.tight_layout(rect=[0,0,1,0.96])

    save_path = SAVE_DIR / f"quantile_{n_q}_hybrid.png"
    plt.savefig(save_path, dpi=300)
    plt.close(fig)

    print(f"Saved figure: {save_path}")


Sectors — U.S — 4 quantiles: avg #months per quantile = {1: np.float64(23.675977653631286), 2: np.float64(30.089385474860336), 3: np.float64(30.93854748603352), 4: np.float64(26.70949720670391)}
Industries — U.S — 4 quantiles: avg #months per quantile = {1: np.float64(23.675977653631286), 2: np.float64(30.089385474860336), 3: np.float64(30.93854748603352), 4: np.float64(26.70949720670391)}
Country Cross Section — America — 4 quantiles: avg #months per quantile = {1: np.float64(23.675977653631286), 2: np.float64(30.089385474860336), 3: np.float64(30.93854748603352), 4: np.float64(26.70949720670391)}
Factors — U.S. — 4 quantiles: avg #months per quantile = {1: np.float64(23.675977653631286), 2: np.float64(30.089385474860336), 3: np.float64(30.93854748603352), 4: np.float64(26.70949720670391)}
Saved figure: C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Research Notebook\Countries\figures\quantile_4_hybrid.png
Sectors — U.S — 3 quantiles: avg #months per quantile = {1: np.float

In [12]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

try:
    import statsmodels.api as sm
    HAVE_SM = True
except Exception:
    HAVE_SM = False


PATH_MACRO = Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Original Reproduction\state_z_america.xlsx")
PATH_FF6_US = Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Original Reproduction\North_America_6_Factors_Daily_clean.xlsx")

DATASETS = [
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Industries\Europe Industries.xlsx"),
     "auto", "Industries — Europe"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Sectors\Per Region\Sector Europe.xlsx"),
     "auto", "Secteur — Europe"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Cross Section Countries\EMEA CS Country.xlsx"),
     "auto", "Country Cross Section — EMEA"),
    (Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Data\Europe\FF6_daily_factors_cleaned.xlsx"),
     "auto", "Factors — Europe)"),
]

OOS_START, OOS_END = pd.Period('2010-01','M'), pd.Period('2024-12','M')  # needs >=5y history before OOS


SAVE_DIR = Path(r"C:\Users\kelvi\Documents\Berkeley\Term 2\Projects\Regimes\Research Notebook\Countries\figures")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

def load_macro_states(path_macro: Path) -> pd.DataFrame:
    Z = pd.read_excel(path_macro)
    if "Month" not in Z.columns:
        for cand in ["month","MONTH","Date","Dates"]:
            if cand in Z.columns:
                Z = Z.rename(columns={cand: "Month"})
                break
    Z["Month"] = pd.to_datetime(Z["Month"], errors="coerce").dt.to_period("M")
    Z = Z.set_index("Month").sort_index()
    macro_cols = [c for c in Z.columns if pd.api.types.is_numeric_dtype(Z[c])]
    return Z[macro_cols].astype(float)

Z_ALL = load_macro_states(PATH_MACRO)

def load_ff6_daily(path: Path) -> pd.DataFrame:
    """
    Returns a DataFrame indexed by daily datetime with columns:
    ['Mkt-RF','SMB','HML','RMW','CMA','Mom','RF'] in DECIMAL (not %).
    Handles YYYYMMDD integer/string dates too.
    """
    df = pd.read_excel(path)
    # find date col
    date_col = next((c for c in ["Date","Dates","DATE","date","dt"] if c in df.columns), None)
    if date_col is None:
        raise ValueError("FF6 file must have a Date column.")
    series = df[date_col]
    looks_8digits = (pd.api.types.is_integer_dtype(series) or series.astype(str).str.fullmatch(r"\d{8}").all())
    if looks_8digits:
        df[date_col] = pd.to_datetime(series.astype(str), format="%Y%m%d", errors="coerce")
    else:
        df[date_col] = pd.to_datetime(series, errors="coerce")

    df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()

    fac_cols = ["Mkt-RF","SMB","HML","RMW","CMA","Mom","RF"]
    present = [c for c in fac_cols if c in df.columns]
    if len(present) < 6:  # need at least the 6 factors + maybe RF
        raise ValueError(f"FF6 file missing expected columns. Found: {present}")

    df[present] = df[present].astype(float)
    if df[present].abs().max().max() > 5:
        df[present] /= 100.0
    return df[present]

FF6 = load_ff6_daily(PATH_FF6_US)

def load_assets(path: Path, kind: str, min_coverage: float = 1.0):

    df = pd.read_excel(path)
    if "Dates" in df.columns:
        df["Dates"] = pd.to_datetime(df["Dates"], errors="coerce")
        df = df.dropna(subset=["Dates"]).set_index("Dates").sort_index()
        assets = [c for c in df.columns if c != "Dates" and pd.api.types.is_numeric_dtype(df[c])]
        if not assets:
            raise ValueError(f"No numeric asset columns in {path.name}")
        sec_ret_daily = df[assets].pct_change(fill_method=None).replace([np.inf,-np.inf], np.nan).dropna(how="all")
        sec_ret_monthly = (1.0 + sec_ret_daily).resample("ME").prod() - 1.0
    else:
        fac_cols = ["Mkt-RF","SMB","HML","RMW","CMA","Mom","RF"]
        date_col = next((c for c in ["Date","Dates","DATE","date","dt"] if c in df.columns), None)
        present = [c for c in fac_cols if c in df.columns]
        if date_col and len(present) >= 5:
            series = df[date_col]
            looks_8digits = (pd.api.types.is_integer_dtype(series) or series.astype(str).str.fullmatch(r"\d{8}").all())
            if looks_8digits:
                df[date_col] = pd.to_datetime(series.astype(str), format="%Y%m%d", errors="coerce")
            else:
                df[date_col] = pd.to_datetime(series, errors="coerce")
            df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
            df[present] = df[present].astype(float)
            if df[present].abs().max().max() > 5:
                df[present] /= 100.0
            assets = [c for c in present if c != "RF"]
            sec_ret_daily = df[assets].replace([np.inf,-np.inf], np.nan).dropna(how="all")
            sec_ret_monthly = (1.0 + sec_ret_daily).resample("ME").prod() - 1.0
        else:
            raise ValueError(f"Unsupported schema in {path.name}")

    need = max(1, int(np.ceil(min_coverage * len(assets))))
    mask_good = sec_ret_monthly.notna().sum(axis=1) >= need
    sec_ret_monthly = sec_ret_monthly.loc[mask_good]
    sector_months = sec_ret_monthly.index.to_period("M")

    keep = sec_ret_daily.index.to_period("M").isin(sector_months)
    sec_ret_daily = sec_ret_daily.loc[keep]
    sec_ret_monthly.index = sec_ret_monthly.index.to_period("M")

    return sec_ret_daily, sec_ret_monthly, assets, sector_months


def ann_sr(s: pd.Series) -> float:
    s = pd.Series(s).dropna()
    if s.empty: return np.nan
    sig = s.std()
    return (s.mean()/sig)*np.sqrt(252) if sig != 0 else np.nan

def rank_similar(F_full: pd.DataFrame, F_train: pd.DataFrame, cur_per: pd.Period):
    """Euclidean on macro states (drop-in: replace with your hybrid if desired)."""
    if cur_per not in F_full.index or F_train.empty:
        return []
    x0 = F_full.loc[[cur_per]].dropna(axis=1, how="any")
    if x0.empty: return []
    Xtr = F_train.dropna(how="any")
    common = Xtr.columns.intersection(x0.columns)
    if common.empty or Xtr.empty: return []
    X = Xtr[common].values.astype(float)
    x = x0.iloc[0][common].values.astype(float)
    d = np.linalg.norm(X - x[None,:], axis=1)   
    return Xtr.index[np.argsort(d)].tolist()


def make_quantiles(ranked_months, n_quantiles: int):
    n = len(ranked_months)
    if n == 0:
        return {q: [] for q in range(1, n_quantiles+1)}
    buckets = np.array_split(ranked_months, n_quantiles)
    return {i+1: list(b) for i, b in enumerate(buckets)}


def build_quantile_series_3q_rolling(path: Path, kind: str):
    n_quantiles = 3
    sec_ret_daily, sec_ret_monthly, assets, sector_months = load_assets(path, kind)
    Z = Z_ALL.reindex(sector_months).ffill()
    F_full = Z.copy()

    oos_stop  = min(OOS_END, sector_months.max())
    months_bt = sector_months[(sector_months >= OOS_START) & (sector_months <= oos_stop)]

    returns_by_q = {q: [] for q in range(1, n_quantiles+1)}

    for i in range(len(months_bt)-1):
        cur_per, nxt_per = months_bt[i], months_bt[i+1]
        start_train = cur_per - 59
        F_train = F_full.loc[(F_full.index >= start_train) & (F_full.index <= cur_per)].dropna(how="any")
        ranked = rank_similar(F_full, F_train, cur_per)
        if not ranked: 
            continue
        buckets = make_quantiles(ranked, n_quantiles)

        dts = sec_ret_daily.index[sec_ret_daily.index.to_period("M") == nxt_per]
        if len(dts) == 0: 
            continue
        Rm = sec_ret_daily.loc[dts, assets]

        for q in range(1, n_quantiles+1):
            idx_sim = pd.PeriodIndex(buckets[q], freq="M") + 1
            idx_sim = sec_ret_monthly.index.intersection(idx_sim)
            if len(idx_sim) == 0:
                q_daily = pd.Series(0.0, index=dts)
            else:
                mu = sec_ret_monthly.loc[idx_sim, assets].mean(axis=0)
                w = np.sign(mu).reindex(assets).fillna(0.0).astype(float)
                q_daily = (Rm * w).mean(axis=1)
            returns_by_q[q].extend(q_daily.items())

    q_series = {q: pd.Series(dict(returns_by_q[q])).sort_index() for q in returns_by_q}
    for q in q_series:
        q_series[q].index = pd.to_datetime(q_series[q].index)

    common_idx = None
    for q in q_series:
        common_idx = q_series[q].index if common_idx is None else common_idx.intersection(q_series[q].index)
    for q in q_series:
        q_series[q] = q_series[q].reindex(common_idx).fillna(0.0)

    long_only = sec_ret_daily[assets].mean(axis=1).reindex(common_idx).fillna(0.0)

    return q_series, long_only


def run_ff6_regression(port_ret: pd.Series, ff6: pd.DataFrame, nw_lags: int = 5):
    """
    Regress *excess* portfolio return (r_p - RF) on (Mkt-RF, SMB, HML, RMW, CMA, Mom).
    Returns dict with alpha, t_alpha, betas, t_betas.
    """
    df = pd.concat([port_ret, ff6], axis=1, join="inner").dropna()
    if df.empty:
        return None

    y = df.iloc[:, 0] - df["RF"]  
    X = df[["Mkt-RF","SMB","HML","RMW","CMA","Mom"]]
    X = sm.add_constant(X) if HAVE_SM else np.c_[np.ones(len(X)), X.values]

    if HAVE_SM:
        model = sm.OLS(y.values, X.values)
        res = model.fit(cov_type="HAC", cov_kwds={"maxlags": nw_lags})
        params = res.params
        tvals  = res.tvalues
        cols   = ["alpha","Mkt-RF","SMB","HML","RMW","CMA","Mom"]
        out = dict(zip(cols, params))
        out.update({f"t_{c}": t for c, t in zip(cols, tvals)})
        return out
    else:
        XtX = X.T @ X
        XtY = X.T @ y.values
        beta = np.linalg.pinv(XtX) @ XtY
        # crude residual std & t-stats (not NW)
        resid = y.values - X @ beta
        n, k = X.shape
        s2 = (resid @ resid) / max(1, n - k)
        cov_beta = s2 * np.linalg.pinv(XtX)
        se = np.sqrt(np.diag(cov_beta))
        tvals = beta / np.where(se == 0, np.nan, se)
        cols = ["alpha","Mkt-RF","SMB","HML","RMW","CMA","Mom"]
        out = dict(zip(cols, beta))
        out.update({f"t_{c}": t for c, t in zip(cols, tvals)})
        return out


all_rows = []
for path, kind, title in DATASETS:
    q_series, _ = build_quantile_series_3q_rolling(path, kind)

    # align with FF6 by date
    for q in [1, 2, 3]:
        r = q_series[q].copy()
        # regression
        res = run_ff6_regression(r, FF6, nw_lags=5)
        if res is None:
            print(f"[WARN] No overlap between {title} Q{q} and FF6 factors.")
            continue

        row = {
            "dataset": title,
            "quantile": q,
            "alpha_bps_per_day": 10000 * res["alpha"],
            "t_alpha": res["t_alpha"],
            "beta_Mkt_RF": res["Mkt-RF"], "t_Mkt_RF": res["t_Mkt-RF"],
            "beta_SMB": res["SMB"],       "t_SMB":   res["t_SMB"],
            "beta_HML": res["HML"],       "t_HML":   res["t_HML"],
            "beta_RMW": res["RMW"],       "t_RMW":   res["t_RMW"],
            "beta_CMA": res["CMA"],       "t_CMA":   res["t_CMA"],
            "beta_Mom": res["Mom"],       "t_Mom":   res["t_Mom"],
            "obs": int(pd.concat([r, FF6["Mkt-RF"]], axis=1, join="inner").dropna().shape[0])
        }
        all_rows.append(row)


results = pd.DataFrame(all_rows)
if not results.empty:
    # Sort & pretty print
    results = results.sort_values(["dataset","quantile"]).reset_index(drop=True)
    pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
    print("\n=== FF6 (incl. Mom) Regressions — Daily, 3 Quantiles, 5y rolling training ===")
    print(results)

    # Save CSV
    out_csv = SAVE_DIR / "europe_3Q_5y_FF6_loadings.csv"
    results.to_csv(out_csv, index=False)
    print(f"\nSaved regression table to: {out_csv}")
else:
    print("No regression rows produced (check date overlaps / inputs).")

try:
    for path, kind, title in DATASETS:
        q_series, _ = build_quantile_series_3q_rolling(path, kind)
        srs = {f"Q{q} SR": ann_sr(q_series[q]) for q in [1,2,3]}
        print(f"{title}: {srs}")
except Exception:
    pass



=== FF6 (incl. Mom) Regressions — Daily, 3 Quantiles, 5y rolling training ===
                         dataset  quantile  alpha_bps_per_day  t_alpha  \
0   Country Cross Section — EMEA         1              3.578    6.020   
1   Country Cross Section — EMEA         2             -0.287   -0.501   
2   Country Cross Section — EMEA         3              1.168    2.017   
3              Factors — Europe)         1              1.563    3.444   
4              Factors — Europe)         2              0.687    1.485   
5              Factors — Europe)         3             -0.494   -1.107   
6            Industries — Europe         1              4.570    3.282   
7            Industries — Europe         2             -0.035   -0.036   
8            Industries — Europe         3             -0.348   -0.371   
9               Secteur — Europe         1              4.018    3.671   
10              Secteur — Europe         2             -0.545   -0.449   
11              Secteur — Europe 